In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append('../src')

from preprocess import encode_profiles, load_profiles
from segment_profiles import cluster_profiles

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
raw_path = Path('../data/raw_profiles.csv')
fallback_path = Path('../data/profiles_template.csv')
source_path = raw_path if raw_path.exists() and raw_path.stat().st_size > 1 else fallback_path

df_raw = load_profiles(source_path)
df_encoded = encode_profiles(df_raw)

print(f'Source utilisee : {source_path}')
print(f'Nombre de profils : {len(df_raw)}')
display(df_raw.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df_raw, x='specialite_dominante', ax=axes[0], palette='Set2')
axes[0].set_title('Repartition par specialite dominante')
axes[0].set_xlabel('Specialite')
axes[0].set_ylabel('Nombre de profils')

sns.countplot(data=df_raw, x='statut', ax=axes[1], palette='Pastel1')
axes[1].set_title('Repartition par statut')
axes[1].set_xlabel('Statut')
axes[1].set_ylabel('Nombre de profils')

plt.tight_layout()
plt.show()

In [ ]:
score_columns = [
    'implication_academique',
    'interet_contenu_pedagogique',
    'implication_vie_etudiante',
    'interet_evenementiel',
    'niveau_technique_estime',
    'aisance_numerique',
    'reactivite_percue',
    'presence_numerique_visible',
]

sns.heatmap(df_encoded[score_columns].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matrice de correlation des variables comportementales')
plt.show()

In [ ]:
df_clustered, model, scaler = cluster_profiles(df_encoded, n_clusters=min(4, len(df_encoded)))

sns.countplot(
    data=df_clustered,
    y='segment_principal',
    order=df_clustered['segment_principal'].value_counts().index,
    palette='magma',
)
plt.title('Repartition des profils par segment principal')
plt.xlabel('Nombre de profils')
plt.ylabel('Segment principal')
plt.show()

display(df_clustered[['id_profil', 'cluster', 'segment_principal', 'score_confiance_segment']].head())